**KUNITZ DOMAIN PROJECT**

In [ ]:
mkdir -p data intermediate results scripts models

Organize your working directory into logical subfolders:

data: input FASTA and annotation files

intermediate: files generated during processing

results: BLAST, HMM outputs, classification sets

scripts: your custom tools

models: HMM profiles

**Extracting Sequences by Species**:
Separate sequences of interest by species for focused analysis.

In [ ]:
grep "Homo sapiens" kunitz_sequences.fasta > human_kunitz_sequences.fasta
grep -v "Homo sapiens" kunitz_sequences.fasta > non_human_kunitz_sequences.fasta

**Filtering Sequences from UniProt or Custom Sources**:

Extract only Homo sapiens sequences using FASTA header patterns.

In [ ]:
gunzip uniprot_sprot.fasta.gz
awk '/^>/ {f=($0 ~ /OS=Homo sapiens/)} f' input.fasta > human_kunitz_sequences.fasta


**Custom Report to FASTA Conversion (PDB)**

In [ ]:
cat rcsb_pdb_custom_report.csv | tr -d '"' \
| awk -F ',' '{if (length($2)>0) {name=$2}; print name, $3, $4, $5}' \
| grep PF00014 \
| awk '{print ">"$1"_"$3; print $2}' > pdb_kunitz_customreported.fasta

Parse and reformat a custom CSV report into FASTA (only domain PF00014 = Kunitz).

**Clustering with CD-HIT (90% identity)**

In [ ]:
cd-hit -i pdb_kunitz_customreported.fasta -o pdb_kunitz_customreported_nr.fasta -c 0.9

Remove redundancy while preserving domain structure for later modeling.

**Filter Specific Sequences (e.g., Remove Outliers)**

In [ ]:
awk '/^>2ODY_E/ {getline; next} {print}' pdb_kunitz_customreported_nr.fasta > pdb_kunitz_customreported_filtered.fasta

Remove sequences that are too long or problematic for MSA/HMM.

**Cluster Parsing and Representative Extraction**

In [ ]:
awk 'BEGIN{skip=0} /^>Cluster 0$/ {skip=1; next} /^>Cluster/ {skip=0} !skip' pdb_kunitz_customreported_nr.fasta.clstr > pdb_kunitz_customreported_filtered.clstr
clstr2txt.pl pdb_kunitz_customreported_filtered.clstr > pdb_kunitz.clusters.txt
awk '$5 == 1 {print $1}' pdb_kunitz.clusters.txt > pdb_kunitz_rp.ids

# Extract representative sequences
for i in $(cat pdb_kunitz_rp.ids); do
  grep -A 1 "^>$i" pdb_kunitz_customreported.fasta | head -n 2 >> pdb_kunitz_rp.fasta
done


**Header Cleanup for Compatibility**

In [ ]:
grep ">" pdb_kunitz_rp.fasta | tr -d ">" | tr "_" ":" > tmp_pdb_efold_ids.txt

Useful for uploading to web tools like EFoldMine or MSA services.

**Format Alignment File for HMMER**

In [ ]:
awk '{
  if (substr($1,1,1)==">") {
    print "\n" toupper($1)
  } else {
    printf "%s", toupper($1)
  }
}' pdb_kunitz_rp.ali > pdb_kunitz_rp_formatted.ali

**Build HMM Profile Using HMMER**

In [ ]:
hmmbuild structural_model.hmm pdb_kunitz_rp_formatted.ali

Create an HMM from aligned Kunitz domain sequences.

**Create BLAST Database & Run Similarity Search**

cat human_kunitz_sequences.fasta non_human_kunitz_sequences.fasta > all_kunitz.fasta
makeblastdb -in all_kunitz.fasta -dbtype prot -out all_kunitz.fasta

blastp -query pdb_kunitz_rp.fasta -db all_kunitz.fasta -out pdb_kunitz_nr_23.blast -outfmt 7

**Filter Hits by Identity/Coverage**

In [ ]:
grep -v "^#" pdb_kunitz_nr_23.blast | awk '{if ($3>=95 && $4>=50) print $2}' | sort -u | cut -d "|" -f 2 > to_remove.ids
grep ">" all_kunitz.fasta | cut -d "|" -f 2 > all_kunitz.id
comm -23 <(sort all_kunitz.id) <(sort to_remove.ids) > to_keep.ids

**Extract Final Positive & Negative Sets**

In [ ]:
python3 get_seq.py to_keep.ids all_kunitz.fasta ok_kunitz.fasta
python3 get_seq.py sp_negs.ids uniprot_sprot.fasta sp_negs.fasta

**Randomize and Split for Training & Testing**

In [ ]:
sort -R sp_negs.ids > random_sp_negs.ids
sort -R to_keep.ids > random_ok_kunitz.ids

head -n 183 random_ok_kunitz.ids > pos_1.ids
tail -n 183 random_ok_kunitz.ids > pos_2.ids

head -n 286417 random_sp_negs.ids > neg_1.ids
tail -n 286417 random_sp_negs.ids > neg_2.ids

**Extract FASTA Sequences for Sets**

In [ ]:
python3 get_seq.py pos_1.ids uniprot_sprot.fasta pos_1.fasta
python3 get_seq.py pos_2.ids uniprot_sprot.fasta pos_2.fasta
python3 get_seq.py neg_1.ids uniprot_sprot.fasta neg_1.fasta
python3 get_seq.py neg_2.ids uniprot_sprot.fasta neg_2.fasta

**Run HMM Search on All Sets**

In [ ]:
hmmsearch -Z 1000 --max --tblout pos_1.out structural_model.hmm pos_1.fasta
hmmsearch -Z 1000 --max --tblout pos_2.out structural_model.hmm pos_2.fasta
hmmsearch -Z 1000 --max --tblout neg_1.out structural_model.hmm neg_1.fasta
hmmsearch -Z 1000 --max --tblout neg_2.out structural_model.hmm neg_2.fasta

**Generate Classification Tables**

In [ ]:
grep -v "^#" pos_1.out | awk '{split($1,a,"|"); print a[2]"\t1\t"$5"\t"$8}' > pos_1.class
grep -v "^#" neg_1.out | awk '{split($1,a,"|"); print a[2]"\t0\t"$5"\t"$8}' > neg_1.class

Add fake negatives with max E-value (unmatched sequences):

In [ ]:
comm -23 <(sort neg_1.ids) <(cut -f1 neg_1.class | sort) | awk '{print $1"\t0\t10.0\t10.0"}' >> neg_1_hits.class

Final sets:

In [ ]:
cat pos_1.class neg_1_hits.class > set_1.class
cat pos_2.class neg_2_hits.class > set_2.class

**Run Performance Evaluation**

In [ ]:
python3 performance.py set_1.class 1e-5

for i in $(seq 1 10); do
  python3 performance.py set_1.class 1e-$i
done | sort -nrk 6

Produces accuracy, precision, recall, MCC, and confusion matrix at multiple thresholds.